# Symmetric regular-solution mixing

## Prerequisites

Review binary mole fractions, ideal configurational entropy, molar enthalpy, and Gibbs free energy.

## Objectives

1. Evaluate unit-aware molar enthalpy, entropy, and Gibbs free energy of mixing.
2. Verify component-exchange symmetry and pure-component limits.
3. Compare the local curvature near equimolar composition at two temperatures.
4. Plot the energetic and entropic contributions without inferring a complete phase diagram.

The interaction parameter and temperatures are illustrative synthetic inputs, not fitted material data.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.thermal.thermo.mixtures.regular_solution import (
    SymmetricRegularSolutionModel,
)
from projectkoios.physkit.units import (
    PhysicalUnit,
    ScalarQuantity,
    Unitless,
    VectorQuantity,
)

## Model

For component-B mole fraction $x$, the symmetric regular-solution model uses

$$
\Delta H_{\mathrm{mix}}=\Omega x(1-x),
$$

$$
\Delta S_{\mathrm{mix}}=-R_g\left[x\ln x+(1-x)\ln(1-x)\right],
$$

and

$$
\Delta G_{\mathrm{mix}}=\Delta H_{\mathrm{mix}}-T\Delta S_{\mathrm{mix}}.
$$

The continuous endpoint limits are zero. Local negative curvature of $\Delta G_{\mathrm{mix}}$ indicates instability with respect to infinitesimal composition fluctuations, but coexistence compositions require a separate common-tangent analysis.


In [ ]:
interaction_parameter_kilojoule_per_mole = 10.0
low_temperature_kelvin = 400.0
high_temperature_kelvin = 800.0
composition_sample_count = 401
molar_energy_unit = PhysicalUnit(expression="kilojoule / mole")

component_b_mole_fraction_values = np.linspace(
    0.0,
    1.0,
    composition_sample_count,
    dtype=np.float64,
)
component_b_mole_fractions = VectorQuantity(
    magnitude=component_b_mole_fraction_values,
    unit=Unitless(),
)
model = SymmetricRegularSolutionModel(
    interaction_parameter=ScalarQuantity(
        magnitude=interaction_parameter_kilojoule_per_mole,
        unit=molar_energy_unit,
    )
)
low_temperature_evaluation = model.evaluate_mixing(
    component_b_mole_fractions=component_b_mole_fractions,
    temperature=ScalarQuantity(
        magnitude=low_temperature_kelvin,
        unit=PhysicalUnit(expression="kelvin"),
    ),
    molar_energy_unit=molar_energy_unit,
)
high_temperature_evaluation = model.evaluate_mixing(
    component_b_mole_fractions=component_b_mole_fractions,
    temperature=ScalarQuantity(
        magnitude=high_temperature_kelvin,
        unit=PhysicalUnit(expression="kelvin"),
    ),
    molar_energy_unit=molar_energy_unit,
)

In [ ]:
endpoint_indices = np.array([0, composition_sample_count - 1], dtype=np.int64)
zero_endpoint_tolerance = 0.0
for evaluation in (low_temperature_evaluation, high_temperature_evaluation):
    assert np.allclose(
        evaluation.molar_enthalpy_of_mixing.magnitude[endpoint_indices],
        0.0,
        rtol=0.0,
        atol=zero_endpoint_tolerance,
    )
    assert np.allclose(
        evaluation.molar_entropy_of_mixing.magnitude[endpoint_indices],
        0.0,
        rtol=0.0,
        atol=zero_endpoint_tolerance,
    )
    assert np.allclose(
        evaluation.molar_gibbs_free_energy_of_mixing.magnitude[endpoint_indices],
        0.0,
        rtol=0.0,
        atol=zero_endpoint_tolerance,
    )
    assert np.allclose(
        evaluation.molar_gibbs_free_energy_of_mixing.magnitude,
        evaluation.molar_gibbs_free_energy_of_mixing.magnitude[::-1],
    )

center_index = composition_sample_count // 2
composition_spacing = (
    component_b_mole_fraction_values[1] - component_b_mole_fraction_values[0]
)


def represented_center_curvature(evaluation):
    gibbs_energy = evaluation.molar_gibbs_free_energy_of_mixing.magnitude
    return (
        gibbs_energy[center_index + 1]
        - 2.0 * gibbs_energy[center_index]
        + gibbs_energy[center_index - 1]
    ) / composition_spacing**2


low_temperature_center_curvature = represented_center_curvature(
    low_temperature_evaluation
)
high_temperature_center_curvature = represented_center_curvature(
    high_temperature_evaluation
)
assert low_temperature_center_curvature < 0.0
assert high_temperature_center_curvature > 0.0

In [ ]:
figure_size_inches = (8.5, 5.2)
figure, axis = plt.subplots(figsize=figure_size_inches)
axis.plot(
    component_b_mole_fraction_values,
    low_temperature_evaluation.molar_enthalpy_of_mixing.magnitude,
    linestyle="--",
    label=r"$\Delta H_{\mathrm{mix}}$",
)
axis.plot(
    component_b_mole_fraction_values,
    -low_temperature_kelvin
    * low_temperature_evaluation.molar_entropy_of_mixing.magnitude,
    linestyle=":",
    label=rf"$-T\Delta S_{{\mathrm{{mix}}}}$ at {low_temperature_kelvin:.0f} K",
)
axis.plot(
    component_b_mole_fraction_values,
    low_temperature_evaluation.molar_gibbs_free_energy_of_mixing.magnitude,
    label=rf"$\Delta G_{{\mathrm{{mix}}}}$ at {low_temperature_kelvin:.0f} K",
)
axis.plot(
    component_b_mole_fraction_values,
    high_temperature_evaluation.molar_gibbs_free_energy_of_mixing.magnitude,
    label=rf"$\Delta G_{{\mathrm{{mix}}}}$ at {high_temperature_kelvin:.0f} K",
)
axis.axhline(0.0, color="black", linewidth=0.8, linestyle="-.")
axis.set(
    xlabel=r"component-B mole fraction $x$",
    ylabel="molar mixing quantity (kJ/mol)",
    title="Synthetic symmetric regular solution",
    xlim=(0.0, 1.0),
)
axis.grid(True)
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

The positive interaction parameter gives an endothermic enthalpy contribution that is largest at equimolar composition. Ideal configurational entropy lowers the Gibbs free energy, with a contribution proportional to temperature. For these illustrative inputs, the low-temperature curve has negative local curvature at $x=1/2$, while the high-temperature curve has positive local curvature there.

Those curvature checks do not by themselves determine coexistence compositions or validate the model for a real mixture.

## Exercises

1. Derive the second derivative of $\Delta G_{\mathrm{mix}}$ at $x=1/2$.
2. Determine the temperature at which that equimolar curvature changes sign.
3. Repeat the calculation for a negative interaction parameter and interpret the change.
4. Implement a separately tested common-tangent analysis before making coexistence claims.
